### continuation de la carto en suivant l'exercice 3-2-2-strategie-manquants

reprise des requetes permettant la génération du dataframe

In [1]:
import duckdb
import pandas as pd
import pyarrow as py
import matplotlib.pyplot as plt

con = duckdb.connect()

In [16]:
con.sql("""
    CREATE TABLE carte_rayon AS
    SELECT
        t.code,
        t.product_name[1].text AS product_name,
        t.brands,
        t.quantity,
        t.categories_tags,
        t.compared_to_category AS main_category,
        t.food_groups_tags[1] AS rayon,
        t.countries_tags,
        t.labels_tags,
        t.stores,
        array_to_string(t.allergens_tags, ',') AS allergens,
        t.ingredients_text[1].text AS ingredients_text,
        t.nova_group,
        t.nutriscore_score,
        t.nutriscore_grade,
        t.environmental_score_grade,
        t.serving_size,
        t.completeness,
        list_filter(t.nutriments, x -> x.name = 'energy-kcal')[1]."100g" AS energy_kcal_100g,
        COALESCE(
            list_filter(t.nutriments, x -> x.name = 'energy-kj')[1]."100g",
            list_filter(t.nutriments, x -> x.name = 'energy')[1]."100g"
        ) AS energy_kj_100g,
        list_filter(t.nutriments, x -> x.name = 'fat')[1]."100g" AS fat_100g,
        list_filter(t.nutriments, x -> x.name = 'saturated-fat')[1]."100g" AS saturated_fat_100g,
        list_filter(t.nutriments, x -> x.name = 'carbohydrates')[1]."100g" AS carbohydrates_100g,
        list_filter(t.nutriments, x -> x.name = 'sugars')[1]."100g" AS sugars_100g,
        list_filter(t.nutriments, x -> x.name = 'fiber')[1]."100g" AS fiber_100g,
        list_filter(t.nutriments, x -> x.name = 'proteins')[1]."100g" AS proteins_100g,
        list_filter(t.nutriments, x -> x.name = 'salt')[1]."100g" AS salt_100g,
        list_filter(t.nutriments, x -> x.name = 'sodium')[1]."100g" AS sodium_100g,
        list_filter(t.nutriments, x -> x.name = 'fruits-vegetables-legumes')[1]."100g" AS fruits_vegetables_legumes_100g
    FROM '../../data/food.parquet' AS t
""")

In [14]:
con.sql("DROP TABLE carte_rayon")

In [ ]:
df = con.sql("SELECT * FROM carte_rayon").df()

### Stratégie de valeurs manquantes par colonne et par usage

| colonne | usage | manquants | décision | pourquoi |
|---|---|---|---|---|
| brands | affichage, substitution (même marque), assistant | 34 % | garder NA + drapeau ; « inconnue » à l'affichage seulement | aucune valeur plausible à inventer |
| categories_tags | substitution, features ML (rayon), assistant | 56 % | règle métier : inclassable si vide et rayon unknown → hors périmètre | sans catégorie, pas de substitution |
| ingredients_text | assistant, features texte | 72 % | garder NA + drapeau | aucune valeur plausible à inventer |
| nova_group | affichage | 75 % | garder NA + drapeau | dérivée des ingrédients, si ingredient absent = 98.5% de chance que nova absent aussi |
| nutriscore_score/_grade | cible du modèle, affichage | 34 % | garder ; « unknown » → NA ; jamais imputer | une cible imputée est une cible inventée |
| energy_kcal_100g | score, affichage, features ML, substitution, assistant | 28 % | imputation métier : kJ / 4,184, sinon 4/4/9 ; reste → médiane par rayon calculée sur le train uniquement, après le split | dérivable avant d'être devinée |
| sugars_100g | score, features ML, affichage, assistant | 32 % | garder NA + drapeau sucres_manquant, jamais imputer | une valeur inventée fausserait le score, manque MAR lié à la complétude de la fiche et au rayon |
| salt_100g / sodium_100g | score, features ML, substitution, affichage, assistant | 37 % | imputation métier : sel = sodium × 2,5 (et inverse) | relation exacte, zéro risque |
| fiber_100g | score, features ML, substitution, affichage, assistant | 62 % | garder NA + drapeau fiber_manquant | MNAR : absence = souvent « pas de fibres » |
| fruit_vegetables_legumes_100g | score, features ML, subsitution, affichage, assistant | 83 % | NA + drapeau, jamais imputer | a limité aux rayons elle est indiqué, sinon traiter NA comme 0 et l'afficher |
| serving_size | sel par portion (affichage) | 68 % | a garder, affichage des valeurs pour 100g si NA | utile quand présente, jamais imputée |
| fat_100g | features ML, contrôle de cohérence, substitution, affichage, assistant | 28 % | garder NA + drapeau fat_manquant ; métier (énergie − 4 × protéines − 4 × glucides) / 9 possible si les 3 sont présentes ; reste → médiane par rayon calculée sur le train uniquement, après le split | MAR : suit la complétude du tableau nutritionnel ; sert de borne haute pour les saturées |
| proteins_100g | score, features ML, affichage, substitution, assistant | 28 % | garder NA + drapeau proteines_manquant ; reste → médiane par rayon calculée sur le train uniquement, après le split ; score non calculable si NA | MAR : manque groupé composante favorable du Nutri-Score |
| carbohydrates_100g | features ML, contrôle de cohérence, substitution, affichage, assistant | 28 % | garder NA + drapeau glucides_manquant ; valeurs > 100 → NA ; sert de borne haute pour sugars_100g ; jamais imputée | utile au profil nutritionnel ; une valeur inventée fausserait les contrôles de cohérence |
| saturated_fat_100g | score, features ML, affichage, substitution, assistant | 34 % | garder NA + drapeau graisses_sat_manquant, règle métier : si fat_100g < 0,5 g → saturées = 0 ; sinon jamais imputer ; score non calculable si NA | composante directe du Nutri-Score, une valeur inventée fausserait le score |
| produit sans aucun nutriment clé | — | 27 % | supprimer la ligne | inutilisable pour le score et la substitution |

### Rédaction des méthodes d'implémentation des décisions : 

Vérification si les unknown sont bien présent sur le parquet aussi

In [3]:
con.sql("""
    SELECT nutriscore_grade, COUNT(*) AS n
    FROM carte_rayon
    GROUP BY 1
    ORDER BY n DESC;
    """)

┌──────────────────┬─────────┐
│ nutriscore_grade │    n    │
│     varchar      │  int64  │
├──────────────────┼─────────┤
│ unknown          │ 3110742 │
│ e                │  387545 │
│ d                │  346120 │
│ c                │  287328 │
│ a                │  200871 │
│ b                │  159205 │
│ not-applicable   │  101213 │
│ NULL             │   43447 │
└──────────────────┴─────────┘

In [8]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE nutriscore_score IS NULL) AS score_null,
        ROUND(100.0 * COUNT(*) FILTER (WHERE nutriscore_score IS NULL) / COUNT(*), 1) AS pct_null
    FROM carte_rayon
""")

┌─────────┬────────────┬──────────┐
│  total  │ score_null │ pct_null │
│  int64  │   int64    │  double  │
├─────────┼────────────┼──────────┤
│ 4636471 │    3255402 │     70.2 │
└─────────┴────────────┴──────────┘

fonction de normalisation pour nutriscore_grade, ici on remplace toute les valeurs unknown ou not-applicable par des pd.Na
la fonction retourne un dataframe nettoyé et modifié (ajout d'une colonne qui indique les grade_non_applicable en true/false) ainsi que le nombre de lignes modifier en unknow et not applicable via un dictionnaire

In [9]:
def normaliser_grade(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """1. Cible : « unknown » et « not-applicable » deviennent NA ; on ne remplit jamais."""
    res = df.copy()
    g = res["nutriscore_grade"].astype("string").str.lower()
    unknown = g.eq("unknown")
    non_appl = g.eq("not-applicable")
    res["grade_non_applicable"] = non_appl.astype("boolean")
    res["nutriscore_grade"] = res["nutriscore_grade"].mask(unknown | non_appl, pd.NA)
    return res, {"unknown": int(unknown.sum()), "not_applicable": int(non_appl.sum())}

la méthode du cours faisait une conversion des valeurs en kj de la colonne vers les kcal si ceux ci étais manquant, hors dans le parquet ce n'est pas le cas la colonne energy-kcal contient uniquement les valeurs en kcal, de ce fait on modifie la create table pour creer une nouvelle colonne energy_kj_100g qui sera la fusion des deux colonnes du parquet : energy-kj et energy

In [11]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE kcal IS NULL) AS kcal_manquants,
        COUNT(*) FILTER (WHERE kcal IS NULL AND kj IS NOT NULL) AS recuperables_kj
    FROM (
        SELECT
            list_filter(t.nutriments, x -> x.name = 'energy-kcal')[1]."100g" AS kcal,
            COALESCE(
                list_filter(t.nutriments, x -> x.name = 'energy-kj')[1]."100g",
                list_filter(t.nutriments, x -> x.name = 'energy')[1]."100g"
            ) AS kj
        FROM '../../data/food.parquet' AS t
    )
""")

┌────────────────┬─────────────────┐
│ kcal_manquants │ recuperables_kj │
│     int64      │      int64      │
├────────────────┼─────────────────┤
│        1291656 │            8011 │
└────────────────┴─────────────────┘

la valeur 900 * 4184 est la conversion en kj pour le maximum possible pour 100gr ( 100 gr de lipides = 900kcal )

In [ ]:
def imputer_energie_metier(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """2. Énergie : d'abord les kJ (exact), ensuite 4/4/9 (approximation), jamais une moyenne."""
    res = df.copy()
    kcal = res["energy_kcal_100g"]

    depuis_kj = kcal.isna() & res["energy_kj_100g"].between(0, (900 * 4.184))
    kcal = kcal.mask(depuis_kj, (res["energy_kj_100g"] / 4.184).round(1))

    calc = 4 * res["carbohydrates_100g"] + 4 * res["proteins_100g"] + 9 * res["fat_100g"]
    depuis_macros = kcal.isna() & calc.notna() & (calc <= 900)
    kcal = kcal.mask(depuis_macros, calc.round(1))

    res["energy_kcal_100g"] = kcal
    return res, {
        "depuis_kj": int(depuis_kj.sum()),
        "depuis_macros": int(depuis_macros.sum()),
        "restant": int(kcal.isna().sum()),
    }

on passe a la méthode imputer_sel_sodium, mais avant on va vérifier si il n'y as pas trop de valeurs aberrante :

In [17]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE salt_100g < 0 OR salt_100g > 100) AS sel_aberrant,
        COUNT(*) FILTER (WHERE sodium_100g < 0 OR sodium_100g > 40) AS sodium_aberrant
    FROM carte_rayon
""")

┌──────────────┬─────────────────┐
│ sel_aberrant │ sodium_aberrant │
│    int64     │      int64      │
├──────────────┼─────────────────┤
│         2543 │            2637 │
└──────────────┴─────────────────┘

ok le résultat est faible et de plus présent surement les même produits ou les deux colonnes sont renseigné, je ne met pas de garde fou dans mon code

In [ ]:
def imputer_sel_sodium(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """3. Sel et sodium : relation exacte dans les deux sens, valeurs plausibles uniquement."""
    res = df.copy()
    sel, sodium = res["salt_100g"], res["sodium_100g"]
    sel_ok = sel.isna() & sodium.between(0, 40)
    sodium_ok = sodium.isna() & sel.between(0, 100)
    res["salt_100g"] = sel.mask(sel_ok, sodium * 2.5)
    res["sodium_100g"] = sodium.mask(sodium_ok, sel / 2.5)
    return res, {
        "sel_depuis_sodium": int(sel_ok.sum()),
        "sodium_depuis_sel": int(sodium_ok.sum()),
        "restant": int(res["salt_100g"].isna().sum()),
    }